### Rebuild it to keep:

issue_description

type

queue

priority

In [2]:
# Imports
import pandas as pd
from pathlib import Path

In [3]:
# Load raw dataset
raw_path = Path("../data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv")

df = pd.read_csv(raw_path)

print("Raw dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Raw dataset shape: (28587, 16)

Columns:
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [4]:
# Preview the data
df.head(5)

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [5]:
# Check missing values
if "language" in df.columns:
    print("Language distribution:\n")
    print(df["language"].value_counts(dropna=False))
else:
    print("No 'language' column found.")

Language distribution:

language
en    16338
de    12249
Name: count, dtype: int64


In [6]:
# Check language distribution
if "language" in df.columns:
    print("Language distribution:\n")
    print(df["language"].value_counts(dropna=False))
else:
    print("No 'language' column found.")

Language distribution:

language
en    16338
de    12249
Name: count, dtype: int64


In [7]:
# Keep only English rows
df_en = df[df["language"] == "en"].copy()

print("English-only dataset shape:", df_en.shape)

English-only dataset shape: (16338, 16)


In [8]:
# Inspect useful columns
useful_cols = ["subject", "body", "priority", "type", "queue", "language"]
existing_cols = [col for col in useful_cols if col in df_en.columns]

df_en[existing_cols].head(10)

,subject,body,priority,type,queue,language
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",high,Incident,Technical Support,en
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",medium,Request,Returns and Exchanges,en
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",low,Request,Billing and Payments,en
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",medium,Problem,Sales and Pre-Sales,en
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",high,Request,Technical Support,en
6,System Interruptions,"Dear Customer Support Team,\n\nI am submitting...",high,Incident,Service Outages and Maintenance,en
7,Connectivity Problems with Printer on MacBook Pro,"Dear Support Team,\n\nI am reporting a recurri...",medium,Incident,Technical Support,en
10,VPN Access Issue,"Customer Support,\n\nWe are encountering a dis...",medium,Incident,Product Support,en
12,Immediate Help Needed: Technical Problem with ...,"Dear Customer Support Team,\n\nI am submitting...",medium,Problem,IT Support,en
13,Inquiry for Detailed Information on Agency Off...,"Dear Customer Support Team,\n\nI hope this mes...",high,Request,Product Support,en


In [9]:
# Combine subject and body into issue_description
df_en["subject"] = df_en["subject"].fillna("").astype(str)
df_en["body"] = df_en["body"].fillna("").astype(str)

df_en["issue_description"] = (
    df_en["subject"].str.strip() + " " + df_en["body"].str.strip()
).str.strip()

df_en[["subject", "body", "issue_description", "priority"]].head(5)

,subject,body,issue_description,priority
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Account Disruption Dear Customer Support Team,...",high
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Query About Smart Home System Integration Feat...,medium
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Inquiry Regarding Invoice Details Dear Custome...,low
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Question About Marketing Agency Software Compa...,medium
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...","Feature Query Dear Customer Support,\n\nI hope...",high


In [10]:
# Build ML dataframe
ml_df = df_en[["issue_description", "type", "queue", "priority"]].copy()

print("Initial ML dataframe shape:", ml_df.shape)
ml_df.head(5)

Initial ML dataframe shape: (16338, 4)


,issue_description,type,queue,priority
1,"Account Disruption Dear Customer Support Team,...",Incident,Technical Support,high
2,Query About Smart Home System Integration Feat...,Request,Returns and Exchanges,medium
3,Inquiry Regarding Invoice Details Dear Custome...,Request,Billing and Payments,low
4,Question About Marketing Agency Software Compa...,Problem,Sales and Pre-Sales,medium
5,"Feature Query Dear Customer Support,\n\nI hope...",Request,Technical Support,high


In [11]:
# Clean ML dataframe
ml_df = ml_df.dropna(subset=["issue_description", "priority"]).copy()

ml_df["issue_description"] = ml_df["issue_description"].astype(str).str.strip()

ml_df["type"] = (
    ml_df["type"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
)

ml_df["queue"] = (
    ml_df["queue"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
)

ml_df["priority"] = (
    ml_df["priority"]
    .astype(str)
    .str.strip()
    .str.upper()
)

ml_df = ml_df[ml_df["issue_description"] != ""].copy()
ml_df = ml_df[ml_df["issue_description"].str.len() > 20].copy()

print("Cleaned shape:", ml_df.shape)

Cleaned shape: (16337, 4)


In [12]:
# Check label distribution
print("Priority distribution (count):\n")
print(ml_df["priority"].value_counts())

print("\nPriority distribution (%):\n")
print((ml_df["priority"].value_counts(normalize=True) * 100).round(2))

Priority distribution (count):

priority
MEDIUM    6617
HIGH      6346
LOW       3374
Name: count, dtype: int64

Priority distribution (%):

priority
MEDIUM    40.50
HIGH      38.84
LOW       20.65
Name: proportion, dtype: float64


In [13]:
# Check uniqueness of issue descriptions
print("Total rows:", len(ml_df))
print("Unique issue_description:", ml_df["issue_description"].nunique())

print("\nTop repeated issue descriptions:\n")
print(ml_df["issue_description"].value_counts().head(10))

Total rows: 16337
Unique issue_description: 16337

Top repeated issue descriptions:

issue_description
Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?                                                                                                                                                                             1
Query About Smart Home System Integration Features Dear Customer Support Team,\n\nI hope this message reaches you well. I am reaching out to request detailed i

In [14]:
# Preview random samples
ml_df.sample(10, random_state=42)

,issue_description,type,queue,priority
24798,Request for Documentation on Integrating Adobe...,REQUEST,PRODUCT SUPPORT,LOW
17458,Problems with Connection Customers are facing ...,PROBLEM,IT SUPPORT,MEDIUM
24918,Seeking Guidance on Securing Medical Data with...,REQUEST,PRODUCT SUPPORT,LOW
25800,Found Issues with Secure Data Access in Hospit...,PROBLEM,TECHNICAL SUPPORT,MEDIUM
19109,Inquiries on ClickUp Billing Plans Hello Custo...,REQUEST,BILLING AND PAYMENTS,MEDIUM
24919,Review and Update Compatibility Settings for E...,CHANGE,IT SUPPORT,MEDIUM
11633,Ensuring Medical Data Security in Healthcare I...,REQUEST,RETURNS AND EXCHANGES,MEDIUM
11104,Report on Billing Mistake A billing error happ...,INCIDENT,BILLING AND PAYMENTS,MEDIUM
22338,Support Required for Hadoop Integration Troubl...,INCIDENT,RETURNS AND EXCHANGES,LOW
15791,Data Security Practices Hello Customer Support...,REQUEST,PRODUCT SUPPORT,MEDIUM


In [15]:
# inspect text lengths
ml_df["text_length"] = ml_df["issue_description"].str.len()

print("Text length summary:\n")
print(ml_df["text_length"].describe())

Text length summary:

count    16337.000000
mean       405.400135
std        179.656656
min         24.000000
25%        255.000000
50%        416.000000
75%        565.000000
max       1189.000000
Name: text_length, dtype: float64


In [16]:
# Filter out extremely short tickets
ml_df = ml_df[ml_df["issue_description"].str.len() > 20].copy()

print("Shape after length filter:", ml_df.shape)

Shape after length filter: (16337, 5)


In [17]:
# Final sanity checks
print("Unique issue descriptions:", ml_df["issue_description"].nunique())
print("Unique types:", ml_df["type"].nunique())
print("Unique queues:", ml_df["queue"].nunique())

print("\nPriority distribution:")
print(ml_df["priority"].value_counts())

Unique issue descriptions: 16337
Unique types: 4
Unique queues: 10

Priority distribution:
priority
MEDIUM    6617
HIGH      6346
LOW       3374
Name: count, dtype: int64


In [18]:
# Save processed ML dataset
from pathlib import Path

processed_path = Path("../data/processed/ml_priority_dataset_stage3.csv")
processed_path.parent.mkdir(parents=True, exist_ok=True)

ml_df_final = ml_df[["issue_description", "type", "queue", "priority"]].copy()

ml_df_final.to_csv(processed_path, index=False)

print(f"Saved Stage 3 dataset to: {processed_path}")
print(ml_df_final.shape)

Saved Stage 3 dataset to: ..\data\processed\ml_priority_dataset_stage3.csv
(16337, 4)


In [19]:
# Reload and verify saved file
check_df = pd.read_csv(processed_path)

print("Reloaded saved dataset shape:", check_df.shape)
print("\nColumns:", check_df.columns.tolist())

print("\nPriority distribution:")
print(check_df["priority"].value_counts())

Reloaded saved dataset shape: (16337, 4)

Columns: ['issue_description', 'type', 'queue', 'priority']

Priority distribution:
priority
MEDIUM    6617
HIGH      6346
LOW       3374
Name: count, dtype: int64
